# SAFE-AI Metrics: binary and multiclass examples

This notebook demonstrates the main public functions of `safe-ai-metrics` for both **binary** and **multiclass** classification.

Main functions covered:

| Family | Functions |
|---|---|
| RGA | `rga_score`, `rga_curve`, `aurga_score`, `compare_rga`, `plot_rga` |
| RGR | `rgr_score`, `rgr_curve`, `aurgr_score`, `compare_rgr`, `plot_rgr` |
| RGE | `rge_score`, `rge_curve`, `aurge_score`, `compare_rge`, `plot_rge` |

The notebook uses small synthetic sklearn datasets so it can run quickly.


## 1. Installation

Run this cell only if `safe-ai-metrics` is not already installed.


In [ ]:
# !pip install safe-ai-metrics

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from safeai.rga import (
    rga_score,
    rga_curve,
    aurga_score,
    compare_rga,
    plot_rga
)

from safeai.rgr import (
    rgr_score,
    rgr_curve,
    aurgr_score,
    compare_rgr,
    plot_rgr
)

from safeai.rge import (
    rge_score,
    rge_curve,
    aurge_score,
    compare_rge,
    plot_rge
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Part A — Binary classification

For binary classification, RGA can be used with a 1D score vector such as the probability of the positive class.
RGR and RGE can use full probability matrices with `class_order=[0, 1]`.


## 3. Binary dataset and models

In [ ]:
x_bin, y_bin = make_classification(
    n_samples=800,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.75, 0.25],
    class_sep=1.0,
    random_state=RANDOM_STATE
)

feature_names_bin = [f'feature_{i}' for i in range(x_bin.shape[1])]

x_bin_train, x_bin_test, y_bin_train, y_bin_test = train_test_split(
    x_bin,
    y_bin,
    test_size=0.3,
    stratify=y_bin,
    random_state=RANDOM_STATE
)

log_reg_bin = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)

rf_bin = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=RANDOM_STATE
)

log_reg_bin.fit(x_bin_train, y_bin_train)
rf_bin.fit(x_bin_train, y_bin_train)

prob_lr_bin = log_reg_bin.predict_proba(x_bin_test)
prob_rf_bin = rf_bin.predict_proba(x_bin_test)

class_order_bin = np.array([0, 1])

score_lr_bin = prob_lr_bin[:, 1]
score_rf_bin = prob_rf_bin[:, 1]

print('Binary probability matrix shape:', prob_lr_bin.shape)
print('Binary class order:', class_order_bin)

## 4. Binary RGA examples

In [ ]:
# 1) Rga_score: one scalar RGA value
rga_lr_bin = rga_score(y_bin_test, score_lr_bin)
rga_rf_bin = rga_score(y_bin_test, score_rf_bin)

print(f'Binary Logistic Regression RGA: {rga_lr_bin:.4f}')
print(f'Binary Random Forest RGA:       {rga_rf_bin:.4f}')

In [ ]:
# 2) Rga_curve: full curve result dictionary
rga_curve_lr_bin = rga_curve(
    y_bin_test,
    score_lr_bin,
    n_segments=10,
    curve_method='auto'
)

print(rga_curve_lr_bin.keys())
print(f"Binary RGA:   {rga_curve_lr_bin['rga']:.4f}")
print(f"Binary AURGA: {rga_curve_lr_bin['aurga_perfect']:.4f}")

In [ ]:
# 3) Aurga_score: only the area under the RGA curve
aurga_lr_bin = aurga_score(
    y_bin_test,
    score_lr_bin,
    n_segments=10,
    curve_method='auto'
)

print(f'Binary Logistic Regression AURGA: {aurga_lr_bin:.4f}')

In [ ]:
# 4) Compare_rga: compare several binary score arrays
rga_results_bin = compare_rga(
    {
        'Logistic Regression': score_lr_bin,
        'Random Forest': score_rf_bin,
    },
    y_bin_test,
    n_segments=10,
    curve_method='partial',
    verbose=True
)

rga_results_bin.keys()

In [ ]:
# 5) Plot_rga: plot comparison
plot_rga(
    rga_results_bin,
    show=True
)

## 5. Binary RGR examples

In [ ]:
# Create one noisy version of the test features for scalar RGR.
rng = np.random.default_rng(RANDOM_STATE)
x_bin_test_noisy = x_bin_test + rng.normal(0.0, 0.05, size=x_bin_test.shape)
prob_lr_bin_noisy = log_reg_bin.predict_proba(x_bin_test_noisy)

# 6) Rgr_score: one scalar RGR value between original and perturbed probabilities
rgr_lr_bin = rgr_score(
    prob_lr_bin,
    prob_lr_bin_noisy,
    class_order=class_order_bin
)

print(f'Binary Logistic Regression RGR under one noise level: {rgr_lr_bin:.4f}')

In [ ]:
# 7) Rgr_curve: robustness curve example under Gaussian noise
noise_strengths = [0.0, 0.02, 0.05, 0.10, 0.15]

rgr_curve_lr_bin = rgr_curve(
    log_reg_bin,
    x_bin_test,
    strengths=noise_strengths,
    method='noise',
    prob_original=prob_lr_bin,
    model_class_order=log_reg_bin.classes_,
    class_order=class_order_bin,
    model_type='sklearn',
    random_seed=RANDOM_STATE,
    verbose=True
)

print(rgr_curve_lr_bin.keys())
print(f"Binary AURGR: {rgr_curve_lr_bin['aurgr']:.4f}")

In [ ]:
# 8) Aurgr_score: only the area under the RGR curve
aurgr_lr_bin = aurgr_score(
    log_reg_bin,
    x_bin_test,
    strengths=noise_strengths,
    method='noise',
    prob_original=prob_lr_bin,
    model_class_order=log_reg_bin.classes_,
    class_order=class_order_bin,
    model_type='sklearn',
    random_seed=RANDOM_STATE,
    verbose=False
)

print(f'Binary Logistic Regression AURGR: {aurgr_lr_bin:.4f}')

In [ ]:
# 9) Compare_rgr: compare robustness curves for several models
rgr_models_bin = {
    'Logistic Regression': (
        log_reg_bin,
        x_bin_test,
        prob_lr_bin,
        log_reg_bin.classes_,
        'sklearn',
        None
    ),
    'Random Forest': (
        rf_bin,
        x_bin_test,
        prob_rf_bin,
        rf_bin.classes_,
        'sklearn',
        None
    )
}

rgr_results_bin = compare_rgr(
    rgr_models_bin,
    strengths=noise_strengths,
    class_order=class_order_bin,
    method='noise',
    random_seed=RANDOM_STATE,
    verbose=True,
    show=False
)

rgr_results_bin.keys()

In [ ]:
# 10) Plot_rgr: plot comparison
plot_rgr(
    rgr_results_bin,
    x_key='noise_levels',
    x_label='Noise standard deviation',
    title='Binary RGR comparison under Gaussian noise',
    show=True
)

## 6. Binary RGE examples

In [ ]:
# Create one reduced feature matrix by masking one feature
x_bin_test_reduced = x_bin_test.copy()
x_bin_test_reduced[:, 0] = 0.0

prob_lr_bin_reduced = log_reg_bin.predict_proba(x_bin_test_reduced)

# 11) Rge_score: one scalar RGE value between full and reduced probabilities
rge_lr_bin = rge_score(
    prob_lr_bin,
    prob_lr_bin_reduced,
    class_order=class_order_bin
)

print(f'Binary Logistic Regression RGE after masking one feature: {rge_lr_bin:.4f}')

In [ ]:
# 12) Rge_curve: tabular feature-removal curve
rge_curve_lr_bin = rge_curve(
    log_reg_bin,
    x_bin_test,
    method='tabular',
    feature_names=feature_names_bin,
    prob_full=prob_lr_bin,
    model_class_order=log_reg_bin.classes_,
    class_order=class_order_bin,
    model_type='sklearn',
    n_steps=5,
    masking_method='greedy',
    baseline='zero',
    verbose=True
)

print(rge_curve_lr_bin.keys())
print(f"Binary AURGE: {rge_curve_lr_bin['aurge']:.4f}")

In [ ]:
# 13) Aurge_score: only the area under the RGE curve
aurge_lr_bin = aurge_score(
    log_reg_bin,
    x_bin_test,
    method='tabular',
    feature_names=feature_names_bin,
    prob_full=prob_lr_bin,
    model_class_order=log_reg_bin.classes_,
    class_order=class_order_bin,
    model_type='sklearn',
    n_steps=5,
    masking_method='greedy',
    baseline='zero',
    verbose=False
)

print(f'Binary Logistic Regression AURGE: {aurge_lr_bin:.4f}')

In [ ]:
# 14) Compare_rge: compare explainability curves for several models
rge_models_bin = {
    'Logistic Regression': (
        log_reg_bin,
        x_bin_test,
        feature_names_bin,
        prob_lr_bin,
        log_reg_bin.classes_,
        'sklearn',
        None
    ),
    'Random Forest': (
        rf_bin,
        x_bin_test,
        feature_names_bin,
        prob_rf_bin,
        rf_bin.classes_,
        'sklearn',
        None
    )
}

rge_results_bin = compare_rge(
    rge_models_bin,
    class_order=class_order_bin,
    method='tabular',
    n_steps=5,
    masking_method='greedy',
    baseline='zero',
    verbose=True,
    show=False
)

rge_results_bin.keys()

In [ ]:
# 15) Plot_rge: plot comparison
plot_rge(
    rge_results_bin,
    x_key='x_axis',
    x_label='Fraction of features removed',
    title='Binary RGE comparison with greedy tabular masking',
    show=True
)

# Part B — Multiclass classification

For multiclass classification, pass the full probability matrix and the model class order.


## 7. Multiclass dataset and models

In [ ]:
x_multi, y_multi = make_classification(
    n_samples=900,
    n_features=10,
    n_informative=7,
    n_redundant=1,
    n_classes=3,
    n_clusters_per_class=1,
    weights=[0.55, 0.30, 0.15],
    class_sep=1.2,
    random_state=RANDOM_STATE
)

feature_names_multi = [f'feature_{i}' for i in range(x_multi.shape[1])]

x_multi_train, x_multi_test, y_multi_train, y_multi_test = train_test_split(
    x_multi,
    y_multi,
    test_size=0.3,
    stratify=y_multi,
    random_state=RANDOM_STATE
)

log_reg_multi = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)

rf_multi = RandomForestClassifier(
    n_estimators=120,
    max_depth=6,
    random_state=RANDOM_STATE
)

log_reg_multi.fit(x_multi_train, y_multi_train)
rf_multi.fit(x_multi_train, y_multi_train)

prob_lr_multi = log_reg_multi.predict_proba(x_multi_test)
prob_rf_multi = rf_multi.predict_proba(x_multi_test)

class_order_multi = np.array([0, 1, 2])

print('Multiclass probability matrix shape:', prob_lr_multi.shape)
print('Multiclass class order:', class_order_multi)

## 8. Multiclass RGA examples

For multiclass RGA, pass the full probability matrix and `class_order`.


In [ ]:
rga_lr_multi = rga_score(
    y_multi_test,
    prob_lr_multi,
    class_order=class_order_multi
)

rga_rf_multi = rga_score(
    y_multi_test,
    prob_rf_multi,
    class_order=class_order_multi
)

print(f'Multiclass Logistic Regression RGA: {rga_lr_multi:.4f}')
print(f'Multiclass Random Forest RGA: {rga_rf_multi:.4f}')

In [ ]:
rga_curve_lr_multi = rga_curve(
    y_multi_test,
    prob_lr_multi,
    class_order=class_order_multi,
    n_segments=10,
    curve_method='auto'
)

print(rga_curve_lr_multi.keys())
print(f"Multiclass RGA: {rga_curve_lr_multi['rga']:.4f}")
print(f"Multiclass AURGA: {rga_curve_lr_multi['aurga']:.4f}")

In [ ]:
aurga_lr_multi = aurga_score(
    y_multi_test,
    prob_lr_multi,
    class_order=class_order_multi,
    n_segments=10,
    curve_method='auto'
)

print(f'Multiclass Logistic Regression AURGA: {aurga_lr_multi:.4f}')

In [ ]:
rga_results_multi = compare_rga(
    {
        'Logistic Regression': (prob_lr_multi, class_order_multi),
        'Random Forest': (prob_rf_multi, class_order_multi),
    },
    y_multi_test,
    n_segments=10,
    curve_method='auto',
    verbose=True
)

plot_rga(
    rga_results_multi,
    show=True
)

## 9. Multiclass RGR examples

The workflow is the same as binary, but `class_order` has three classes and probabilities have three columns.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
x_multi_test_noisy = x_multi_test + rng.normal(0.0, 0.05, size=x_multi_test.shape)
prob_lr_multi_noisy = log_reg_multi.predict_proba(x_multi_test_noisy)

rgr_lr_multi = rgr_score(
    prob_lr_multi,
    prob_lr_multi_noisy,
    class_order=class_order_multi
)

print(f'Multiclass Logistic Regression RGR under one noise level: {rgr_lr_multi:.4f}')

In [ ]:
rgr_curve_lr_multi = rgr_curve(
    log_reg_multi,
    x_multi_test,
    strengths=noise_strengths,
    method='noise',
    prob_original=prob_lr_multi,
    model_class_order=log_reg_multi.classes_,
    class_order=class_order_multi,
    model_type='sklearn',
    random_seed=RANDOM_STATE,
    verbose=True
)

print(rgr_curve_lr_multi.keys())
print(f"Multiclass AURGR: {rgr_curve_lr_multi['aurgr']:.4f}")

In [ ]:
aurgr_lr_multi = aurgr_score(
    log_reg_multi,
    x_multi_test,
    strengths=noise_strengths,
    method='noise',
    prob_original=prob_lr_multi,
    model_class_order=log_reg_multi.classes_,
    class_order=class_order_multi,
    model_type='sklearn',
    random_seed=RANDOM_STATE,
    verbose=False,
)

print(f'Multiclass Logistic Regression AURGR: {aurgr_lr_multi:.4f}')

In [ ]:
rgr_models_multi = {
    'Logistic Regression': (
        log_reg_multi,
        x_multi_test,
        prob_lr_multi,
        log_reg_multi.classes_,
        'sklearn',
        None,
    ),
    'Random Forest': (
        rf_multi,
        x_multi_test,
        prob_rf_multi,
        rf_multi.classes_,
        'sklearn',
        None,
    ),
}

rgr_results_multi = compare_rgr(
    rgr_models_multi,
    strengths=noise_strengths,
    class_order=class_order_multi,
    method='noise',
    random_seed=RANDOM_STATE,
    verbose=True,
    show=False,
)

plot_rgr(
    rgr_results_multi,
    x_key='noise_levels',
    x_label='Noise standard deviation',
    title='Multiclass RGR comparison under Gaussian noise',
    show=True,
)

## 10. Multiclass RGE examples

For multiclass RGE, use full probability matrices and the same tabular masking workflow.


In [ ]:
x_multi_test_reduced = x_multi_test.copy()
x_multi_test_reduced[:, 0] = 0.0

prob_lr_multi_reduced = log_reg_multi.predict_proba(x_multi_test_reduced)

rge_lr_multi = rge_score(
    prob_lr_multi,
    prob_lr_multi_reduced,
    class_order=class_order_multi
)

print(f'Multiclass Logistic Regression RGE after masking one feature: {rge_lr_multi:.4f}')

In [ ]:
rge_curve_lr_multi = rge_curve(
    log_reg_multi,
    x_multi_test,
    method='tabular',
    feature_names=feature_names_multi,
    prob_full=prob_lr_multi,
    model_class_order=log_reg_multi.classes_,
    class_order=class_order_multi,
    model_type='sklearn',
    n_steps=5,
    masking_method='greedy',
    baseline='zero',
    verbose=True
)

print(rge_curve_lr_multi.keys())
print(f"Multiclass AURGE: {rge_curve_lr_multi['aurge']:.4f}")

In [ ]:
aurge_lr_multi = aurge_score(
    log_reg_multi,
    x_multi_test,
    method='tabular',
    feature_names=feature_names_multi,
    prob_full=prob_lr_multi,
    model_class_order=log_reg_multi.classes_,
    class_order=class_order_multi,
    model_type='sklearn',
    n_steps=5,
    masking_method='greedy',
    baseline='zero',
    verbose=False
)

print(f'Multiclass Logistic Regression AURGE: {aurge_lr_multi:.4f}')

In [ ]:
rge_models_multi = {
    'Logistic Regression': (
        log_reg_multi,
        x_multi_test,
        feature_names_multi,
        prob_lr_multi,
        log_reg_multi.classes_,
        'sklearn',
        None
    ),
    'Random Forest': (
        rf_multi,
        x_multi_test,
        feature_names_multi,
        prob_rf_multi,
        rf_multi.classes_,
        'sklearn',
        None
    )
}

rge_results_multi = compare_rge(
    rge_models_multi,
    class_order=class_order_multi,
    method='tabular',
    n_steps=5,
    masking_method='greedy',
    baseline='zero',
    verbose=True,
    show=False
)

plot_rge(
    rge_results_multi,
    x_key='x_axis',
    x_label='Fraction of features removed',
    title='Multiclass RGE comparison with greedy tabular masking',
    show=True
)